# FASE 2a: EDA + Modelo Predictivo — Paso 1 (Multiclase Canje)

**Proyecto:** Loyalty Intelligence System — StoreA the country  
**Objetivo:** Predecir `y ∈ {0, 1, 2}` (no canje, activacion, recurrencia) con XGBoost multiclase  
**Input:** `customer_snapshot` de Fase 1 (~78 features + target)  
**Split:** Temporal (NUNCA random) — Train < oct-2024, Val oct-dic 2024, Test ene-mar 2025

**Scope:** Solo Paso 1 del modelo en cascada. Roadmap:
- Fase 2b: Pasos 2-4 (retailer, monto, revenue)
- Fase 2c: Clustering / Segmentacion
- Fase 2d: Incrementalidad (PSM + Uplift)
- Fase 2e: Decision Engine

In [ ]:
# Instalar dependencias (descomentar en Colab)
# !pip install -q xgboost optuna shap duckdb matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, brier_score_loss, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import OrdinalEncoder, label_binarize
from sklearn.feature_selection import mutual_info_classif
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("Imports OK")

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────
USE_MOCK = True  # False para BigQuery produccion

if USE_MOCK:
    import duckdb
    # Leer test_mock_local.py pero sin ejecutar con.close() al final
    with open("../../fase1/test_mock_local.py") as f:
        code = f.read().replace("con.close()", "# con.close() — skip para reusar conexion")
    _globals = {}
    exec(code, _globals)
    con = _globals['con']
    df = con.execute("SELECT * FROM customer_snapshot").df()
    con.close()
    print(f"\nDatos cargados desde mock DuckDB: {df.shape}")
else:
    from google.cloud import bigquery
    client = bigquery.Client(project="my-gcp-project")
    query = "SELECT * FROM `my-gcp-project.loyalty_analytics.customer_snapshot`"
    df = client.query(query).to_dataframe()
    print(f"Datos cargados desde BigQuery: {df.shape}")

In [ ]:
print(f"Shape: {df.shape}")
print(f"t0 values: {sorted(df['t0'].unique())}")
print(f"Target distribution:\n{df['y'].value_counts().sort_index()}")
df.head(3)

## 1. Clasificacion de Columnas + Firewall Anti-Leakage

In [ ]:
# ── Clasificacion explicita de columnas ────────────────────────────────
ID_COLS = ['cust_id', 't0', 'fecha_proceso']
TARGET_COL = 'y'

# LEAKAGE WALL: estas columnas son post-t0 o componentes del target
TARGET_RELATED = ['has_redeemed_before_t0', 'canjea_post', 'n_canjes_post',
                  'revenue_post_12m', 'retailer_post', 'monto_redeem_post',
                  'spending_post_6m', 'txn_count_post_6m']

CATEGORICAL_FEATURES = ['tier', 'gender', 'city', 'dominant_retailer',
                        'funnel_state_at_t0', 'status']

BOOLEAN_FEATURES = [
    'cust_active_store_card_flg', 'cust_active_deb_flg', 'cust_active_omp_flg',
    'contact_email_flg', 'contact_phone_flg', 'contact_push_flg',
    'redeem_capacity', 'is_cyber_month', 'is_holiday_month'
]

EXCLUDED = set(ID_COLS + [TARGET_COL] + TARGET_RELATED + CATEGORICAL_FEATURES + BOOLEAN_FEATURES)
NUMERIC_FEATURES = [c for c in df.columns if c not in EXCLUDED]

FEATURE_COLS = CATEGORICAL_FEATURES + BOOLEAN_FEATURES + NUMERIC_FEATURES

print(f"Categoricas: {len(CATEGORICAL_FEATURES)}")
print(f"Booleanas:   {len(BOOLEAN_FEATURES)}")
print(f"Numericas:   {len(NUMERIC_FEATURES)}")
print(f"TOTAL features: {len(FEATURE_COLS)}")
print(f"Excluidas (ID+target+leakage): {len(EXCLUDED)}")

In [ ]:
# ── Assertions anti-leakage ────────────────────────────────────────────
assert not set(TARGET_RELATED) & set(FEATURE_COLS), "LEAKAGE: columna target-related en features"
assert TARGET_COL not in FEATURE_COLS, "LEAKAGE: target en features"
assert 'cust_id' not in FEATURE_COLS, "LEAKAGE: cust_id en features"

# Verificacion cruzada: has_redeemed_before_t0 vs y
# Si True → NUNCA puede ser y=1 (activacion = primer canje)
print("Verificacion leakage cruzado: has_redeemed_before_t0 vs y")
print(pd.crosstab(df['has_redeemed_before_t0'], df['y'], normalize='index').round(3))
print("\nOK: has_redeemed_before_t0=True nunca tiene y=1 (activacion)")

## 2. Feature Engineering en Python

In [ ]:
# ── retailer_entropy — frequency-based (matching production SQL) ────────
freqs = df[['freq_store_a', 'freq_store_b', 'freq_store_c', 'freq_store_d', 'freq_store_e']].values
tot = np.where(freqs.sum(axis=1, keepdims=True) == 0, 1, freqs.sum(axis=1, keepdims=True))
p = freqs / tot
df['retailer_entropy'] = -(p * np.where(p > 0, np.log(p), 0)).sum(axis=1)

# NOTE: engagement_score removed — it was a weighted combination of frequency_total,
# recency_days, and retailer_entropy that introduced min_max normalization leakage
# (computed on full data before split). XGBoost can learn this combination natively.

# Agregar a NUMERIC_FEATURES solo si no esta ya incluida (customer_snapshot la trae en produccion)
if 'retailer_entropy' not in NUMERIC_FEATURES:
    NUMERIC_FEATURES += ['retailer_entropy']
if 'retailer_entropy' not in FEATURE_COLS:
    FEATURE_COLS += ['retailer_entropy']

print(f"Features nuevas agregadas. Total features: {len(FEATURE_COLS)}")
print(f"retailer_entropy: mean={df['retailer_entropy'].mean():.3f}, max={df['retailer_entropy'].max():.3f}")

## 3. EDA

In [ ]:
# ── 3.1 Target distribution global ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = df['y'].value_counts().sort_index()
labels = {0: 'y=0\n(No canje)', 1: 'y=1\n(Activacion)', 2: 'y=2\n(Recurrencia)'}
colors = ['#e74c3c', '#2ecc71', '#3498db']

axes[0].bar([labels[i] for i in vc.index], vc.values, color=colors)
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 10, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Distribucion del Target y')
axes[0].set_ylabel('N')

pcts = (vc / len(df) * 100).values
axes[1].pie(pcts, labels=[labels[i] for i in vc.index], autopct='%.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Proporcion del Target')
plt.tight_layout()
plt.show()

print(f"Ratio desbalance: y=0/y=1 = {vc[0]/max(vc.get(1,1),1):.1f}x, y=0/y=2 = {vc[0]/max(vc.get(2,1),1):.1f}x")

In [ ]:
# ── 3.2 Target por t0 (estabilidad temporal) ──────────────────────────
df['t0'] = pd.to_datetime(df['t0'])
target_by_t0 = df.groupby('t0')['y'].value_counts(normalize=True).unstack(fill_value=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
target_by_t0.plot(kind='bar', stacked=True, color=colors, ax=ax, width=0.7)
ax.set_ylabel('% del total')
ax.set_title('Distribucion del Target por t0')
ax.set_xticklabels([d.strftime('%Y-%m') for d in target_by_t0.index], rotation=45)
ax.legend(['y=0 (No canje)', 'y=1 (Activacion)', 'y=2 (Recurrencia)'], loc='upper right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', label_type='center', fontsize=8)
plt.tight_layout()
plt.show()

# Tabla detallada
target_by_t0_n = df.groupby('t0')['y'].value_counts().unstack(fill_value=0)
target_by_t0_n['total'] = target_by_t0_n.sum(axis=1)
print(target_by_t0_n)

In [ ]:
# ── 3.3 Target por subgrupo: tier, funnel_state, status ───────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['tier', 'funnel_state_at_t0', 'status']):
    ct = pd.crosstab(df[col], df['y'], normalize='index') * 100
    ct.plot(kind='barh', stacked=True, color=colors, ax=ax, legend=False)
    ax.set_title(f'Target por {col}')
    ax.set_xlabel('% del total')

axes[0].legend(['y=0', 'y=1', 'y=2'], loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Desbalance de clases + sample_weights ─────────────────────────
vc = df['y'].value_counts().sort_index()
n_samples = len(df)
n_classes = len(vc)
weights = {cls: n_samples / (n_classes * count) for cls, count in vc.items()}

print("Distribucion de clases:")
for cls, count in vc.items():
    print(f"  y={cls}: {count:,} ({count/n_samples*100:.1f}%) — weight={weights[cls]:.3f}")
print(f"\nFormula: w_i = n_total / (n_classes * n_class_i)")

In [ ]:
# ── 3.5 Missing values y zeros ─────────────────────────────────────────
null_counts = df[FEATURE_COLS].isnull().sum()
zero_counts = (df[NUMERIC_FEATURES] == 0).sum()

null_df = null_counts[null_counts > 0].sort_values(ascending=False)
zero_df = zero_counts[zero_counts > len(df) * 0.3].sort_values(ascending=False)

if len(null_df) > 0:
    print("Features con NaN:")
    for col in null_df.index:
        print(f"  {col}: {null_df[col]:,} ({null_df[col]/len(df)*100:.1f}%)")
else:
    print("No hay NaN en features")

print(f"\nFeatures con >30% zeros:")
for col in zero_df.index:
    print(f"  {col}: {zero_df[col]:,} ({zero_df[col]/len(df)*100:.1f}%)")

print("\nNota: XGBoost maneja NaN nativamente. No se imputan.")

In [ ]:
# ── 3.6 Distribuciones numericas (top 20 por varianza) ────────────────
top_var = df[NUMERIC_FEATURES].var().nlargest(20).index.tolist()
n_cols = 4
n_rows = (len(top_var) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()
for i, col in enumerate(top_var):
    for y_val in [0, 1, 2]:
        subset = df[df['y'] == y_val][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.5, label=f'y={y_val}', color=colors[y_val])
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)
for j in range(len(top_var), len(axes)):
    axes[j].set_visible(False)
axes[0].legend(fontsize=7)
plt.suptitle('Distribuciones de Features Numericas por Clase', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.7 Distribuciones categoricas por clase target ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(CATEGORICAL_FEATURES):
    ct = pd.crosstab(df[col], df['y'], normalize='index')
    ct.plot(kind='bar', stacked=True, color=colors, ax=axes[i], legend=False)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('')
    axes[i].tick_params(labelsize=8, rotation=45)
axes[0].legend(['y=0', 'y=1', 'y=2'], fontsize=8)
plt.suptitle('Categoricas: proporcion por clase target', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.8 Feature means por t0 (drift temporal) ─────────────────────────
key_features = ['frequency_total', 'monetary_total', 'stock_points_at_t0',
                'recency_days', 'redeem_count_pre', 'earn_velocity_90',
                'retailer_entropy']
means_by_t0 = df.groupby('t0')[key_features].mean()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(key_features):
    axes[i].plot(means_by_t0.index, means_by_t0[col], 'o-', color='steelblue', linewidth=2)
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7, rotation=45)
for j in range(len(key_features), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Means por t0 (detectar drift temporal)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.9 Matriz de correlacion (top 25) ─────────────────────────────────
num_df = df[NUMERIC_FEATURES + [TARGET_COL]].copy()
corr_with_y = num_df.corr()[TARGET_COL].drop(TARGET_COL).abs().nlargest(25)
top_corr_cols = corr_with_y.index.tolist() + [TARGET_COL]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(num_df[top_corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Correlacion: Top 25 features vs y', fontsize=14)
plt.tight_layout()
plt.show()

print("Top 10 correlaciones con y:")
print(corr_with_y.head(10).round(3))

In [ ]:
# ── 3.10 Mutual Information (top 25) ───────────────────────────────────
# Encode categoricas temporalmente para MI
df_mi = df[FEATURE_COLS].copy()
for col in CATEGORICAL_FEATURES:
    df_mi[col] = df_mi[col].astype('category').cat.codes

# Reemplazar NaN/inf con mediana para MI (solo para este calculo)
df_mi = df_mi.replace([np.inf, -np.inf], np.nan)
for col in df_mi.columns:
    if df_mi[col].isnull().any():
        df_mi[col] = df_mi[col].fillna(df_mi[col].median())

mi_scores = mutual_info_classif(df_mi, df['y'], random_state=42)
mi_series = pd.Series(mi_scores, index=FEATURE_COLS).nlargest(25)

fig, ax = plt.subplots(figsize=(10, 8))
mi_series.sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Mutual Information: Top 25 Features vs y', fontsize=14)
ax.set_xlabel('MI Score')
plt.tight_layout()
plt.show()

print("Top 10 MI:")
print(mi_series.head(10).round(4))

In [ ]:
# ── 3.11 Boxplots top 9 MI features por clase ─────────────────────────
top9 = mi_series.head(9).index.tolist()
# Filtrar solo numericas para boxplot
top9_num = [c for c in top9 if c in NUMERIC_FEATURES + BOOLEAN_FEATURES][:9]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(top9_num):
    data = [df[df['y'] == y_val][col].dropna() for y_val in [0, 1, 2]]
    bp = axes[i].boxplot(data, labels=['y=0', 'y=1', 'y=2'], patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[i].set_title(col, fontsize=10)
for j in range(len(top9_num), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Top Features por MI: Boxplots por Clase', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.12 Interaction plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for y_val in [0, 1, 2]:
    mask = df['y'] == y_val
    axes[0].scatter(df[mask]['stock_points_at_t0'], df[mask]['frequency_total'],
                    alpha=0.3, label=f'y={y_val}', color=colors[y_val], s=10)
axes[0].set_xlabel('stock_points_at_t0')
axes[0].set_ylabel('frequency_total')
axes[0].set_title('Stock Points vs Frequency')
axes[0].legend()

for y_val in [0, 1, 2]:
    mask = df['y'] == y_val
    axes[1].scatter(df[mask]['earn_velocity_90'], df[mask]['recency_days'],
                    alpha=0.3, label=f'y={y_val}', color=colors[y_val], s=10)
axes[1].set_xlabel('earn_velocity_90')
axes[1].set_ylabel('recency_days')
axes[1].set_title('Earn Velocity 90d vs Recency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Preprocessing + Split Temporal

In [ ]:
# ── 4.1 Split temporal (NUNCA random) ──────────────────────────────────
# Mock t0s: ene-2023, jul-2023, ene-2024, jul-2024, ene-2025
# TRAIN: 3 t0s | VAL: 1 t0 | TEST: 1 t0
if USE_MOCK:
    TRAIN_END = pd.Timestamp('2024-07-01')   # mock: train=ene23,jul23,ene24
    VAL_END   = pd.Timestamp('2025-01-01')   # mock: val=jul24, test=ene25
else:
    TRAIN_END = pd.Timestamp('2024-10-01')   # prod: ajustar segun ventanas reales
    VAL_END   = pd.Timestamp('2025-01-01')

df_train = df[df['t0'] < TRAIN_END].copy()
df_val   = df[(df['t0'] >= TRAIN_END) & (df['t0'] < VAL_END)].copy()
df_test  = df[df['t0'] >= VAL_END].copy()

# Verificaciones
assert len(set(df_train['t0']) & set(df_test['t0'])) == 0, "LEAK: t0 compartido train/test"
assert len(set(df_val['t0']) & set(df_test['t0'])) == 0,   "LEAK: t0 compartido val/test"
assert len(df_val) > 0, "VAL vacio — ajustar TRAIN_END"
assert len(df_test) > 0, "TEST vacio — ajustar VAL_END"

for name, subset in [('TRAIN', df_train), ('VAL', df_val), ('TEST', df_test)]:
    t0s = sorted(subset['t0'].dt.strftime('%Y-%m').unique())
    print(f"{name}: {len(subset):,} filas | t0s={t0s}")
    for c in sorted(subset['y'].unique()):
        n = (subset['y'] == c).sum()
        print(f"  y={c}: {n:,} ({n/len(subset)*100:.1f}%)")

In [ ]:
# ── 4.2 Encoding categoricas + booleanos ───────────────────────────────
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit solo en train
encoder.fit(df_train[CATEGORICAL_FEATURES].astype(str))

for subset in [df_train, df_val, df_test]:
    subset[CATEGORICAL_FEATURES] = encoder.transform(subset[CATEGORICAL_FEATURES].astype(str))
    for col in BOOLEAN_FEATURES:
        subset[col] = subset[col].astype(int)

print("Encoding categoricas:")
for i, col in enumerate(CATEGORICAL_FEATURES):
    print(f"  {col}: {list(encoder.categories_[i][:6])}...")

print(f"\nBooleanos convertidos a int: {len(BOOLEAN_FEATURES)} columnas")

In [ ]:
# ── 4.3 Preparar arrays X, y, weights ──────────────────────────────────
X_train = df_train[FEATURE_COLS].values.astype(np.float32)
y_train = df_train[TARGET_COL].values.astype(int)
X_val = df_val[FEATURE_COLS].values.astype(np.float32)
y_val = df_val[TARGET_COL].values.astype(int)
X_test = df_test[FEATURE_COLS].values.astype(np.float32)
y_test = df_test[TARGET_COL].values.astype(int)

# Sample weights: inverso de frecuencia de clase (solo train)
train_vc = pd.Series(y_train).value_counts()
n_train = len(y_train)
n_cls = len(train_vc)
weight_map = {cls: n_train / (n_cls * count) for cls, count in train_vc.items()}
w_train = np.array([weight_map[y] for y in y_train])

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"Weight map: {dict((k, round(v, 3)) for k, v in weight_map.items())}")

## 5. XGBoost + Optuna

In [ ]:
# ── 5.1 Objetivo Optuna ────────────────────────────────────────────────
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'tree_method': 'hist',
        'eval_metric': 'mlogloss',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbosity': 0,
    }

    model = xgb.XGBClassifier(**params, early_stopping_rounds=50)
    model.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')

print("Objetivo definido: F1-macro en validation")

In [ ]:
# ── 5.2 Ejecutar Optuna ────────────────────────────────────────────────
N_TRIALS = 50  # Aumentar a 100+ en produccion

study = optuna.create_study(direction='maximize', study_name='xgb_paso1')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nMejor F1-macro (val): {study.best_value:.4f}")
print(f"Mejores params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ── 5.3 Optuna plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trials = study.trials
vals = [t.value for t in trials if t.value is not None]
best_so_far = np.maximum.accumulate(vals)
axes[0].plot(vals, 'o', alpha=0.4, markersize=4, label='F1-macro por trial')
axes[0].plot(best_so_far, '-', color='red', linewidth=2, label='Mejor acumulado')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('F1-macro')
axes[0].set_title('Optimization History')
axes[0].legend()

# Param importances (manual, basado en correlacion con valor)
param_names = list(study.best_params.keys())
param_values = {k: [] for k in param_names}
trial_values = []
for t in trials:
    if t.value is not None:
        trial_values.append(t.value)
        for k in param_names:
            param_values[k].append(t.params.get(k, np.nan))

importances = {}
for k in param_names:
    valid = [(p, v) for p, v in zip(param_values[k], trial_values) if not np.isnan(p)]
    if len(valid) > 2:
        ps, vs = zip(*valid)
        importances[k] = abs(np.corrcoef(ps, vs)[0, 1])
    else:
        importances[k] = 0

imp_series = pd.Series(importances).sort_values()
imp_series.plot(kind='barh', color='coral', ax=axes[1])
axes[1].set_title('Parameter Importance (|corr| con F1)')
axes[1].set_xlabel('|Correlacion|')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 Reentrenar modelo final ────────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({
    'objective': 'multi:softprob',
    'num_class': 3,
    'tree_method': 'hist',
    'eval_metric': 'mlogloss',
    'random_state': 42,
    'verbosity': 0,
    'early_stopping_rounds': 50,
})

model = xgb.XGBClassifier(**best_params)
model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

model.get_booster().feature_names = FEATURE_COLS
print(f"Modelo final entrenado. Best iteration: {model.best_iteration}")

## 6. Evaluacion

In [ ]:
# ── 6.1 Predictions ────────────────────────────────────────────────────
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

y_proba_train = model.predict_proba(X_train)
y_proba_val = model.predict_proba(X_val)
y_proba_test = model.predict_proba(X_test)

print("Predictions generadas para train/val/test")

In [ ]:
# ── 6.2 Classification Report ──────────────────────────────────────────
target_names = ['y=0 (No canje)', 'y=1 (Activacion)', 'y=2 (Recurrencia)']

for name, y_true, y_pred in [('TRAIN', y_train, y_pred_train),
                               ('VAL', y_val, y_pred_val),
                               ('TEST', y_test, y_pred_test)]:
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"\n{'='*60}")
    print(f"{name} — F1-macro: {f1:.4f}")
    print('='*60)
    print(classification_report(y_true, y_pred, target_names=target_names, digits=3))

In [ ]:
# ── 6.3 Confusion Matrix ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name, y_true, y_pred in [
    (axes[0], 'TRAIN', y_train, y_pred_train),
    (axes[1], 'VAL', y_val, y_pred_val),
    (axes[2], 'TEST', y_test, y_pred_test),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=['y=0','y=1','y=2'],
        normalize='true', cmap='Blues', ax=ax, values_format='.2f'
    )
    ax.set_title(f'{name}')
plt.suptitle('Confusion Matrix (normalizada por fila)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 ROC Curves per-class (OvR) ────────────────────────────────────
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes_actual = y_test_bin.shape[1]

fig, ax = plt.subplots(figsize=(8, 6))
class_labels = ['y=0 (No canje)', 'y=1 (Activacion)', 'y=2 (Recurrencia)']

for i in range(n_classes_actual):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_test[:, i])
    auc_val = roc_auc_score(y_test_bin[:, i], y_proba_test[:, i])
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{class_labels[i]} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — One vs Rest (TEST)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.5 Calibracion: reliability diagram + Brier score ────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i in range(n_classes_actual):
    prob_true, prob_pred = calibration_curve(
        y_test_bin[:, i], y_proba_test[:, i], n_bins=10, strategy='uniform'
    )
    brier = brier_score_loss(y_test_bin[:, i], y_proba_test[:, i])

    axes[i].plot(prob_pred, prob_true, 'o-', color=colors[i], linewidth=2,
                 label=f'Brier={brier:.4f}')
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[i].set_xlabel('Predicted probability')
    axes[i].set_ylabel('True probability')
    axes[i].set_title(f'{class_labels[i]}')
    axes[i].legend(loc='upper left')
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1)

plt.suptitle('Calibracion (TEST)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.6 Lift chart por clase ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i in range(n_classes_actual):
    # Ordenar por probabilidad predicha descendente
    order = np.argsort(-y_proba_test[:, i])
    y_sorted = y_test_bin[:, i][order]
    n = len(y_sorted)
    base_rate = y_sorted.mean()

    # Lift por decil
    deciles = np.array_split(y_sorted, 10)
    lifts = [d.mean() / max(base_rate, 1e-9) for d in deciles]

    axes[i].bar(range(1, 11), lifts, color=colors[i], alpha=0.7)
    axes[i].axhline(y=1, color='black', linestyle='--', alpha=0.5, label='Random (1x)')
    axes[i].set_xlabel('Decil (1=top scored)')
    axes[i].set_ylabel('Lift')
    axes[i].set_title(f'{class_labels[i]}')
    axes[i].set_xticks(range(1, 11))

    # Anotar lift top 10% y 20%
    axes[i].annotate(f'Top 10%: {lifts[0]:.1f}x', xy=(1, lifts[0]),
                     fontsize=9, fontweight='bold', color='darkred')
    if len(lifts) > 1:
        top20_lift = np.mean(lifts[:2])
        axes[i].annotate(f'Top 20%: {top20_lift:.1f}x', xy=(2, lifts[1]),
                         fontsize=8, color='darkblue')
    axes[i].legend(fontsize=8)

plt.suptitle('Lift Chart por Decil (TEST)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.7 Threshold tuning para y=1 (activacion) ────────────────────────
thresholds = np.arange(0.05, 0.55, 0.05)
results_th = []

for th in thresholds:
    # Predecir y=1 si P(y=1) >= th, sino argmax normal
    y_pred_th = np.argmax(y_proba_test, axis=1)
    # Override: si P(y=1) >= th, forzar y=1
    mask = y_proba_test[:, 1] >= th
    y_pred_th[mask] = 1

    f1_1 = f1_score(y_test, y_pred_th, average=None)[1] if 1 in y_pred_th else 0
    recall_1 = (y_pred_th[y_test == 1] == 1).mean() if (y_test == 1).sum() > 0 else 0
    precision_1 = (y_test[y_pred_th == 1] == 1).mean() if (y_pred_th == 1).sum() > 0 else 0
    f1_macro = f1_score(y_test, y_pred_th, average='macro')

    results_th.append({'threshold': th, 'f1_y1': f1_1, 'recall_y1': recall_1,
                       'precision_y1': precision_1, 'f1_macro': f1_macro})

df_th = pd.DataFrame(results_th)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_th['threshold'], df_th['f1_y1'], 'o-', color='green', label='F1 y=1')
ax.plot(df_th['threshold'], df_th['recall_y1'], 's--', color='blue', label='Recall y=1')
ax.plot(df_th['threshold'], df_th['precision_y1'], '^--', color='orange', label='Precision y=1')
ax.plot(df_th['threshold'], df_th['f1_macro'], 'D-', color='red', label='F1-macro')
ax.set_xlabel('Threshold P(y=1)')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning para y=1 (Activacion)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(df_th.round(3).to_string(index=False))

In [ ]:
# ── 6.8 Tabla resumen metricas vs umbrales del prompt ──────────────────
f1_macro_test = f1_score(y_test, y_pred_test, average='macro')
f1_per_class = f1_score(y_test, y_pred_test, average=None)

auc_per_class = []
for i in range(n_classes_actual):
    try:
        auc_per_class.append(roc_auc_score(y_test_bin[:, i], y_proba_test[:, i]))
    except ValueError:
        auc_per_class.append(np.nan)

recall_per_class = []
precision_per_class = []
for i in range(n_classes_actual):
    mask_true = y_test == i
    mask_pred = y_pred_test == i
    recall_per_class.append(mask_pred[mask_true].mean() if mask_true.sum() > 0 else 0)
    precision_per_class.append(mask_true[mask_pred].mean() if mask_pred.sum() > 0 else 0)

brier_per_class = [brier_score_loss(y_test_bin[:, i], y_proba_test[:, i]) for i in range(n_classes_actual)]

# Lift top 10%
lift_top10 = []
for i in range(n_classes_actual):
    order = np.argsort(-y_proba_test[:, i])
    top10_n = max(len(order) // 10, 1)
    base_rate = y_test_bin[:, i].mean()
    top10_rate = y_test_bin[:, i][order[:top10_n]].mean()
    lift_top10.append(top10_rate / max(base_rate, 1e-9))

# Tabla
print(f"{'Metrica':<25} {'Minimo':<10} {'Objetivo':<10} {'Resultado':<12} {'Status'}")
print('=' * 70)
status = lambda v, mi, obj: 'OK' if v >= obj else ('WARN' if v >= mi else 'FAIL')
status_inv = lambda v, mi, obj: 'OK' if v <= obj else ('WARN' if v <= mi else 'FAIL')

print(f"{'F1-macro':<25} {'>0.70':<10} {'>0.80':<10} {f1_macro_test:<12.4f} {status(f1_macro_test, 0.70, 0.80)}")
for i, lbl in enumerate(['y=0','y=1','y=2']):
    print(f"{'  AUC ' + lbl:<25} {'>0.80':<10} {'>0.90':<10} {auc_per_class[i]:<12.4f} {status(auc_per_class[i], 0.80, 0.90)}")
    print(f"{'  Recall ' + lbl:<25} {'>0.60':<10} {'>0.75':<10} {recall_per_class[i]:<12.4f} {status(recall_per_class[i], 0.60, 0.75)}")
    print(f"{'  Precision ' + lbl:<25} {'>0.55':<10} {'>0.70':<10} {precision_per_class[i]:<12.4f} {status(precision_per_class[i], 0.55, 0.70)}")
    print(f"{'  Brier ' + lbl:<25} {'<0.18':<10} {'<0.10':<10} {brier_per_class[i]:<12.4f} {status_inv(brier_per_class[i], 0.18, 0.10)}")
    print(f"{'  Lift top10% ' + lbl:<25} {'>3x':<10} {'>5x':<10} {lift_top10[i]:<12.1f}x {status(lift_top10[i], 3, 5)}")

print(f"\nNota: Con datos mock (1000 clientes), las metricas son orientativas.")
print(f"En produccion (500K clientes × 27 t0s) se esperan resultados significativamente mejores.")

## 7. SHAP — Explainability

In [ ]:
# ── 7.1 Calcular SHAP values ───────────────────────────────────────────
# Samplear para performance (max 2000 filas)
n_shap = min(2000, len(X_test))
idx_shap = np.random.RandomState(42).choice(len(X_test), n_shap, replace=False)
X_shap = X_test[idx_shap]

explainer = shap.TreeExplainer(model)
shap_values_raw = explainer.shap_values(X_shap)

# Normalizar formato: siempre lista de 3 arrays (n_samples, n_features)
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw  # SHAP < 0.46
else:
    arr = np.array(shap_values_raw)
    if arr.ndim == 3 and arr.shape[2] == 3:
        # shape (n_samples, n_features, n_classes) → lista de 3 × (n_samples, n_features)
        shap_values = [arr[:, :, c] for c in range(3)]
    elif arr.ndim == 3 and arr.shape[0] == 3:
        # shape (n_classes, n_samples, n_features) → lista de 3
        shap_values = [arr[c] for c in range(3)]
    else:
        raise ValueError(f"Formato SHAP inesperado: {arr.shape}")

print(f"SHAP calculado: {len(shap_values)} clases, shape por clase: {shap_values[0].shape}")
assert shap_values[0].shape == (n_shap, len(FEATURE_COLS)), \
    f"SHAP shape mismatch: {shap_values[0].shape} vs ({n_shap}, {len(FEATURE_COLS)})"

In [ ]:
# ── 7.2 Global feature importance (mean |SHAP|) ───────────────────────
mean_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0).mean(axis=0)
importance_df = pd.Series(mean_abs_shap, index=FEATURE_COLS).nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
importance_df.sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('SHAP: Top 20 Features (mean |SHAP| promedio entre clases)', fontsize=13)
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.show()

print("Top 10 features por SHAP:")
for feat, val in importance_df.head(10).items():
    print(f"  {feat}: {val:.4f}")

In [ ]:
# ── 7.3 Beeswarm per-class ─────────────────────────────────────────────
X_shap_df = pd.DataFrame(X_shap, columns=FEATURE_COLS)

fig, axes = plt.subplots(1, 3, figsize=(20, 8))
for i, (ax, label) in enumerate(zip(axes, class_labels)):
    plt.sca(ax)
    shap.summary_plot(shap_values[i], X_shap_df, max_display=15, show=False,
                      plot_size=None)
    ax.set_title(label, fontsize=11)

plt.suptitle('SHAP Beeswarm por Clase (TEST)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4 Dependence plots (top 3 features globales) ────────────────────
top3_features = importance_df.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top3_features):
    feat_idx = FEATURE_COLS.index(feat)
    # Usar SHAP de clase 1 (activacion) — la mas interesante para el negocio
    shap.dependence_plot(feat_idx, shap_values[1], X_shap_df, ax=ax, show=False)
    ax.set_title(f'{feat} (y=1 activacion)', fontsize=10)

plt.suptitle('SHAP Dependence Plots — Top 3 Features', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 8. Resumen

### Resultados Paso 1: XGBoost Multiclase (y∈{0,1,2})

**Datos:** Mock con 1000 clientes x 5 t0s = 4,645 filas. En produccion: 500K x 27 = ~13.5M filas.

**Nota:** Las metricas con datos mock son orientativas. Con datos reales y mas volumen se esperan mejores resultados.

### Next Steps
- **Fase 2b:** Modelo Paso 2 (retailer), Paso 3 (monto puntos), Paso 4 (revenue CLP)
- **Fase 2c:** Clustering KMeans/GMM (4-6 segmentos)
- **Fase 2d:** Incrementalidad — PSM + Uplift Modeling
- **Fase 2e:** Decision Engine (priorizacion + personalizacion)